# GIAI ĐOẠN 1: HUẤN LUYỆN BYTE-FALLBACK BPE TOKENIZER (VOCAB 8K) & CHUẨN BỊ DỮ LIỆU

> **Dự án:** Large Language Model from Scratch (Clean Architecture)  
> **Mục tiêu kỹ thuật chuẩn mực:**  
> 1. Tải 500.000 mẫu câu chuyện từ `roneneldan/TinyStories`.  
> 2. **Phân tách thực sự ngẫu nhiên theo Seed 42:** Dùng hoán vị ngẫu nhiên xác định (`permutation`) để chia **475.000 Train / 25.000 Validation**, lưu danh sách chỉ số indices để tái lập bất biến xuyên suốt các giai đoạn.  
> 3. **Không rò rỉ dữ liệu (No Data Leakage):** BPE Tokenizer **chỉ học merge rules trên 475K Train stories**, tập Validation tuyệt đối không tham gia học.  
> 4. **Cơ chế Artifact Lock:** Khóa an toàn chống chạy lại làm ghi đè mất dữ liệu.  
> 5. **Kiểm thử nghiệm thu chặt chẽ (Hard Assertions):** ID liên tục 0..8191, không trùng ID, kiểm tra merge rules hợp lệ, đối chứng SHA256 Fingerprint toàn vẹn.  
> 6. **Fused Streaming Tokenization & Profiling:** Duyệt 1 lượt duy nhất qua buffer ghi thẳng file nhị phân `train.bin`, `val.bin` (`uint16`) và đo lường 100% ngữ liệu.

## 1. Thiết lập Môi trường & Clone Repository Tự động

In [ ]:
# ==============================================================================
# 1. THIẾT LẬP MÔI TRƯỜNG CLOUD & CLONE REPO BẮT BUỘC
# ==============================================================================
import os
import sys

if not os.path.exists('/kaggle/working/Prj_LLM') and os.path.exists('/kaggle/working'):
    %cd /kaggle/working
    !git clone -b dev https://github.com/TyTrinh/Prj_LLM.git
    %cd /kaggle/working/Prj_LLM
    !pip install -q -e .
elif os.path.exists('/kaggle/working/Prj_LLM'):
    %cd /kaggle/working/Prj_LLM
    !git pull origin dev

REPO_ROOT = os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f'📂 Thư mục làm việc: {REPO_ROOT}')
!pip install -q datasets regex wandb matplotlib
print('✅ Thiết lập môi trường thành công!')


## 2. Kết nối Weights & Biases (Dự án: `LLM_BaseModel_TinyStories`)

In [ ]:
# ==============================================================================
# KẾT NỐI VỚI WEIGHTS & BIASES (W&B) QUA KAGGLE SECRETS AN TOÀN
# ==============================================================================
import os
import wandb

# Tên Project và Run chuyên biệt cho giai đoạn Tokenizer & Data Preparation
WANDB_PROJECT = 'LLM_Stage1_Tokenizer_BPE8K'
WANDB_RUN_NAME = 'Tokenizer-BPE8K-TinyStories-500K'

USE_WANDB = True
WANDB_MODE = 'online'  # 'online', 'offline', hoặc 'disabled'
WANDB_API_KEY = None

# 1. Đọc an toàn từ Kaggle Secrets (Add-ons -> Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    WANDB_API_KEY = user_secrets.get_secret('WANDB_API_KEY')
    print('🔑 Đã lấy thành công WANDB_API_KEY từ Kaggle Secrets!')
except Exception:
    WANDB_API_KEY = os.environ.get('WANDB_API_KEY')

if USE_WANDB and WANDB_MODE == 'online':
    try:
        if WANDB_API_KEY:
            wandb.login(key=WANDB_API_KEY)
        else:
            wandb.login()
        print(f'✅ W&B Login thành công! Dự án Tokenizer: {WANDB_PROJECT}')
    except Exception as e:
        print(f'⚠️ Không thể login W&B: {e}. Chuyển sang offline mode.')
        WANDB_MODE = 'offline'


## 3. Tải Dữ liệu & Phân tách Ngẫu nhiên Xác định theo Seed 42

In [ ]:
import json
import numpy as np
from datasets import load_dataset

NUM_SAMPLES = 500_000
TRAIN_RATIO = 0.95
SEED = 42

print(f'📥 Đang tải {NUM_SAMPLES:,} mẫu từ roneneldan/TinyStories...')
dataset = load_dataset('roneneldan/TinyStories', split=f'train[:{NUM_SAMPLES}]')
total_loaded = len(dataset)
print(f'✅ Đã tải thành công: {total_loaded:,} stories.')

# Shuffle ngẫu nhiên thực sự theo seed 42 qua default_rng
rng = np.random.default_rng(seed=SEED)
shuffled_indices = rng.permutation(total_loaded)

num_train = int(total_loaded * TRAIN_RATIO)  # 475,000
train_indices = shuffled_indices[:num_train]
val_indices = shuffled_indices[num_train:]

print(f'• Train stories (Học Tokenizer & Pretrain): {len(train_indices):,}')
print(f'• Validation stories (Đánh giá độc lập): {len(val_indices):,}')

os.makedirs('data', exist_ok=True)
splits_info = {
    'dataset': 'roneneldan/TinyStories',
    'total': total_loaded,
    'train_samples': len(train_indices),
    'val_samples': len(val_indices),
    'seed': SEED,
    'train_indices_preview': train_indices[:20].tolist(),
    'val_indices_preview': val_indices[:20].tolist(),
}
with open('data/splits_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(splits_info, f, indent=2)
print('✅ Đã lưu splits_metadata.json để đảm bảo tính bất biến sang các giai đoạn sau!')


## 4. Huấn luyện BPE Tokenizer (Chốt an toàn Artifact Lock & Vocab 8K)

In [ ]:
import time
from src.tokenizer import BytePairTokenizer

TARGET_VOCAB_SIZE = 8192
OUTPUT_DIR = 'checkpoints/tokenizer_8k'
VOCAB_FILE = os.path.join(OUTPUT_DIR, 'vocab.json')
MERGES_FILE = os.path.join(OUTPUT_DIR, 'merges.txt')

# CỜ KHÓA AN TOÀN (Đặt FORCE_OVERWRITE = True nếu bạn muốn ép huấn luyện lại)
FORCE_OVERWRITE = False

tokenizer = BytePairTokenizer(vocab_size=TARGET_VOCAB_SIZE)

if os.path.exists(VOCAB_FILE) and os.path.exists(MERGES_FILE) and not FORCE_OVERWRITE:
    print('🔒 [ARTIFACT LOCKED] Phát hiện bộ Tokenizer 8K đã tồn tại!')
    print(f'• Đang nạp từ điển có sẵn: {VOCAB_FILE}...')
    tokenizer.load(VOCAB_FILE, MERGES_FILE)
    print(f'✅ Nạp thành công! Vocab size: {len(tokenizer.vocab):,}, Merges: {len(tokenizer.merges):,}')
else:
    print(f'🚀 Bắt đầu huấn luyện BPE Tokenizer trên {len(train_indices):,} câu chuyện Train...')
    train_texts = (dataset[int(i)]['text'] for i in train_indices)
    t0 = time.time()
    tokenizer.train(train_texts, verbose=True)
    t1 = time.time()
    print(f'\n🎉 Huấn luyện hoàn tất trong {t1 - t0:.2f}s ({(t1 - t0)/60:.2f} phút)!')
    
    tokenizer.save(OUTPUT_DIR)
    tokenizer.save('checkpoints')  # Lưu bản mặc định
    
    fingerprint = tokenizer.get_fingerprint()
    meta = {
        'vocab_size': len(tokenizer.vocab),
        'target_vocab_size': TARGET_VOCAB_SIZE,
        'num_merges': len(tokenizer.merges),
        'fingerprint': fingerprint,
        'special_tokens': tokenizer.special_tokens,
        'seed': SEED,
        'train_samples': len(train_indices),
    }
    with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)
    print(f'✅ Đã lưu artifacts và metadata tại {OUTPUT_DIR}')
    print(f'🔒 SHA256 Fingerprint: {fingerprint}')


## 5. Bộ Kiểm Thử Nghiệm Thu Chặt Chẽ (Hard Acceptance Assertions)
Kiểm tra tính toàn vẹn tuyệt đối: ID liên tục 0..8191, không trùng ID, nạp instance mới so sánh Fingerprint và kiểm thử Round-trip.

In [ ]:
# ==============================================================================
# BỘ KIỂM THỬ NGHIỆM THU TOKENIZER (ACCEPTANCE CRITERIA ASSERTIONS)
# ==============================================================================
print('🧪 ĐANG CHẠY BỘ KIỂM THỬ NGHIỆM THU TOÀN VẸN...')

# 1. Kích thước Vocab chính xác 8192
assert len(tokenizer.vocab) == TARGET_VOCAB_SIZE, f'Vocab size {len(tokenizer.vocab)} != {TARGET_VOCAB_SIZE}'

# 2. Dải ID liên tục 100% từ 0 đến 8191
assert sorted(tokenizer.vocab.keys()) == list(range(TARGET_VOCAB_SIZE)), 'Dải ID không liên tục 0..8191!'

# 3. Special tokens khóa cứng từ 0..4
assert set(tokenizer.special_tokens.values()) == set(range(5)), 'Special token IDs không chuẩn 0..4!'

# 4. Raw bytes chiếm dải 5..260
for b in range(256):
    assert tokenizer.vocab[5 + b] == bytes([b]), f'Raw byte {b} sai lệch!'

# 5. Số lượng merge operations chính xác 7.931 (từ 261 đến 8191)
assert len(tokenizer.merges) == TARGET_VOCAB_SIZE - 261, 'Số lượng merge không khớp 7931!'
assert min(tokenizer.merges.values()) == 261 and max(tokenizer.merges.values()) == TARGET_VOCAB_SIZE - 1

# 6. Khởi tạo instance mới, nạp lại artifact từ ổ đĩa và đối chứng Fingerprint
fresh_tokenizer = BytePairTokenizer()
fresh_tokenizer.load(VOCAB_FILE, MERGES_FILE)
assert fresh_tokenizer.get_fingerprint() == tokenizer.get_fingerprint(), 'Fingerprint sau khi nạp không khớp!'

# 7. Kiểm tra decode() ném lỗi khi gặp ID ngoài dải (Không được nuốt lỗi pass)
try:
    fresh_tokenizer.decode([99999])
    assert False, 'Decode không ném ngoại lệ khi gặp ID không hợp lệ!'
except ValueError:
    pass  # Đúng yêu cầu

# 8. Kiểm tra Round-trip trên các mẫu câu đa dạng (Hard Assertions)
test_suite = [
    'Once upon a time, a kind girl named Lily found a magic key in the garden.',
    '"Watch out for the dragon!" shouted Tom as he jumped over the river.',
    '<|user|>Tell me a story about friendship.<|assistant|>Two bears shared their honey.<|eos|>',
    'Trường Đại học Bách Khoa - Đại học Đà Nẵng: Mô hình LLM 8K 🇻🇳🚀',
    'Special chars: math ∑(x_i) >= 0, code def foo(): return None',
]
for s in test_suite:
    enc = fresh_tokenizer.encode(s)
    dec = fresh_tokenizer.decode(enc)
    assert dec == s, f'Lỗi Round-trip thất bại tại: {repr(s)}'

print('=' * 75)
print('🎉 100% TIÊU CHÍ NGHIỆM THU TOKENIZER ĐẠT CHUẨN XUẤT SẮC!')
print(f'• SHA256 Fingerprint bất biến: {fresh_tokenizer.get_fingerprint()}')
print('=' * 75)


## 6. Fused Streaming Tokenization & Profiling Toàn bộ 500.000 Mẫu
Chỉ duyệt 1 lượt duy nhất qua buffer: vừa đo lường 100% ngữ liệu vừa đóng gói `train.bin` và `val.bin` (`uint16`).

In [ ]:
# ==============================================================================
# STREAMING BUFFER TOKENIZATION & 100% CORPUS PROFILING
# ==============================================================================
train_bin_path = 'data/train.bin'
val_bin_path = 'data/val.bin'
eos_id = tokenizer.special_tokens.get('<|eos|>', 2)

char_lengths = np.zeros(total_loaded, dtype=np.int32)
word_lengths = np.zeros(total_loaded, dtype=np.int32)
token_lengths = np.zeros(total_loaded, dtype=np.int32)

# 1. Ghi Train Set theo streaming chunks
print(f'⚡ [1/2] Đang Stream Tokenize tập Train ({len(train_indices):,} câu chuyện)...')
total_train_tokens = 0
with open(train_bin_path, 'wb') as f_train:
    buf = []
    for count, idx in enumerate(train_indices):
        text = dataset[int(idx)]['text']
        char_lengths[idx] = len(text)
        word_lengths[idx] = len(text.split())
        
        tok_ids = tokenizer.encode(text) + [eos_id]
        token_lengths[idx] = len(tok_ids)
        buf.extend(tok_ids)
        total_train_tokens += len(tok_ids)
        
        # Ghi chunk 500K tokens để bảo vệ RAM
        if len(buf) >= 500_000:
            np.array(buf, dtype=np.uint16).tofile(f_train)
            buf.clear()
            
        if (count + 1) % 100_000 == 0:
            print(f'  -> Đã xử lý {count + 1:,}/{len(train_indices):,} câu chuyện Train...')
            
    if buf:
        np.array(buf, dtype=np.uint16).tofile(f_train)
        buf.clear()

# 2. Ghi Val Set theo streaming chunks
print(f'\n⚡ [2/2] Đang Stream Tokenize tập Validation ({len(val_indices):,} câu chuyện)...')
total_val_tokens = 0
with open(val_bin_path, 'wb') as f_val:
    buf = []
    for count, idx in enumerate(val_indices):
        text = dataset[int(idx)]['text']
        char_lengths[idx] = len(text)
        word_lengths[idx] = len(text.split())
        
        tok_ids = tokenizer.encode(text) + [eos_id]
        token_lengths[idx] = len(tok_ids)
        buf.extend(tok_ids)
        total_val_tokens += len(tok_ids)
        
        if len(buf) >= 200_000:
            np.array(buf, dtype=np.uint16).tofile(f_val)
            buf.clear()
            
    if buf:
        np.array(buf, dtype=np.uint16).tofile(f_val)
        buf.clear()

# 3. Báo cáo thống kê toàn diện 100% dataset
print('\n' + '=' * 80)
print(f'📊 BÁO CÁO THỐNG KÊ TOÀN CỤC 100% TẬP DỮ LIỆU ({total_loaded:,} STORIES):')
print('=' * 80)
print(f'• Độ dài Ký tự:       Mean = {np.mean(char_lengths):.1f} | Min = {np.min(char_lengths)} | Max = {np.max(char_lengths)}')
print(f'• Độ dài Số từ:       Mean = {np.mean(word_lengths):.1f} | Min = {np.min(word_lengths)} | Max = {np.max(word_lengths)}')
print(f'• Độ dài Tokens:      Mean = {np.mean(token_lengths):.1f} | Median = {np.median(token_lengths):.1f} | Max = {np.max(token_lengths)}')
print(f'• Phân vị Tokens:     P50 = {np.percentile(token_lengths, 50):.0f} | P90 = {np.percentile(token_lengths, 90):.0f} | P95 = {np.percentile(token_lengths, 95):.0f} | P99 = {np.percentile(token_lengths, 99):.0f}')
print(f'• Tỷ lệ Nén BPE 8K:   {np.sum(token_lengths) / np.sum(word_lengths):.2f} tokens/từ')
print(f'• File train.bin:     {total_train_tokens:,} tokens ({os.path.getsize(train_bin_path)/(1024**2):.2f} MB)')
print(f'• File val.bin:       {total_val_tokens:,} tokens ({os.path.getsize(val_bin_path)/(1024**2):.2f} MB)')
print('=' * 80)
print('🎉 QUY TRÌNH CHUẨN BỊ TOKENIZER & DỮ LIỆU ĐÃ HOÀN TẤT TUYỆT ĐỐI!')

# Gửi metrics thống kê phân phối của Tokenizer & Dataset lên W&B Dashboard
if USE_WANDB:
    run = wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME, mode=WANDB_MODE, reinit=True)
    run.summary['vocab_size'] = TARGET_VOCAB_SIZE
    run.summary['num_merges'] = len(tokenizer.merges)
    run.summary['fingerprint'] = fresh_tokenizer.get_fingerprint()
    run.summary['total_samples'] = total_loaded
    run.summary['train_tokens'] = int(total_train_tokens)
    run.summary['val_tokens'] = int(total_val_tokens)
    run.summary['mean_token_length'] = float(np.mean(token_lengths))
    run.summary['p99_token_length'] = float(np.percentile(token_lengths, 99))
    run.summary['compression_ratio'] = float(np.sum(token_lengths) / np.sum(word_lengths))
    print(f'📊 Đã gửi toàn bộ thông số phân phối và artifact fingerprint lên W&B Project: {WANDB_PROJECT}!')
    run.finish()
